In [46]:
# Standard library imports
import os
import sys
import math
import random
from collections import Counter, defaultdict
from typing import List
from random import sample as random_sample

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from tqdm import tqdm

# PyTorch imports
import torch
import torch.nn as nn
from torch import Tensor, no_grad, argmax
from torch import max as torch_max
from torch.nn import CrossEntropyLoss
from torch.optim import SGD, AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, CosineAnnealingWarmRestarts
from torch.utils.data import Dataset, DataLoader, random_split, WeightedRandomSampler
from torchsummary import summary
import torchvision.transforms as transforms
from torchvision.transforms import functional as F

# Ignite imports
from ignite.engine import Events, create_supervised_trainer, create_supervised_evaluator
from ignite.handlers import ReduceLROnPlateauScheduler
from ignite.metrics import Accuracy, Fbeta, Loss, Precision, Recall

In [47]:
# transforming

class RandomAdjustColor:
    def __init__(self, brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1):
        self.brightness = brightness
        self.contrast = contrast
        self.saturation = saturation
        self.hue = hue

    def __call__(self, img):
        if random.random() < 0.5:
            img = F.adjust_brightness(img, random.uniform(1 - self.brightness, 1 + self.brightness))
        if random.random() < 0.5:
            img = F.adjust_contrast(img, random.uniform(1 - self.contrast, 1 + self.contrast))
        if random.random() < 0.5:
            img = F.adjust_saturation(img, random.uniform(1 - self.saturation, 1 + self.saturation))
        if random.random() < 0.5:
            img = F.adjust_hue(img, random.uniform(-self.hue, self.hue))
        return img


def add_gaussian_noise(tensor, mean=0.0, std=0.05):
    return tensor + torch.randn_like(tensor) * std


# Basic transformation without augmentation
basic_transformation = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Basic data augmentation
basic_augmentation = transforms.Compose([
    # Random horizontal flip
    transforms.RandomHorizontalFlip(p=0.5),

    # Random rotation (up to 15 degrees)
    transforms.RandomRotation(degrees=15),

    # Random resizing and cropping
    transforms.RandomResizedCrop(size=(288, 512), scale=(0.8, 1.0), ratio=(0.75, 1.33)),

    # Random color adjustments
    RandomAdjustColor(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),

    # Convert to PyTorch tensor
    transforms.ToTensor(),

    # Image normalization (standard ImageNet values)
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Advanced data augmentation pipeline
advanced_augmentation = transforms.Compose([
    # 1. Rotation: Random rotation between -30° and +30°
    transforms.RandomRotation(degrees=(-30, 30)),

    # 2. Horizontal flip: Flip image horizontally
    transforms.RandomHorizontalFlip(p=0.5),

    # 3. Resizing: Standard size for network input
    transforms.Resize((224, 224)),

    # 4. Translation: Random shifts along X and Y axes
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),

    # 5. Brightness adjustment: Random brightness change [0.7, 1.3]
    transforms.ColorJitter(brightness=(0.7, 1.3)),

    # 6. Contrast adjustment: Random contrast change [0.8, 1.2]
    transforms.ColorJitter(contrast=(0.8, 1.2)),

    # 7. Saturation adjustment: Random saturation change [0.5, 1.5]
    transforms.ColorJitter(saturation=(0.5, 1.5)),

    # Convert to PyTorch tensor (must be before Lambda)
    transforms.ToTensor(),

    # 8. Noise injection: Add Gaussian noise
    transforms.Lambda(add_gaussian_noise),

    # 9. Perspective distortion: Random perspective transformation
    transforms.RandomPerspective(distortion_scale=0.2, p=0.5),

    # 10. Cropping: Random crop with padding
    transforms.RandomCrop(size=(224, 224), padding=10),

    # 11. Blurring: Apply Gaussian blur
    transforms.GaussianBlur(kernel_size=(5, 5), sigma=(0.1, 2.0)),

    # 12. Sharpness adjustment: Reduce image quality
    transforms.RandomAdjustSharpness(sharpness_factor=0.5, p=0.5),

    # 13. Random erasing: Simulate shadows or occlusions
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.1), ratio=(0.3, 3.3), value=0),

    # Image normalization (required for most neural networks)
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [48]:
# PeopleDataset from dataloader

class PeopleDataset(Dataset):
    def __init__(self, data_dir, transform=None):
        self.data_dir = data_dir
        self.transform = transform
        self.labels_df = pd.read_csv(os.path.join(data_dir, 'train_answers.csv'))
        self.img_dir = os.path.join(data_dir, 'img_train')
        self.image_files = []
        self.labels = []
        self.class_names = ['sports', 'inactivity quiet/light', 'miscellaneous', 'occupation', 'water activities',
                            'home activities', 'lawn and garden', 'religious activities', 'winter activities',
                            'conditioning exercise', 'bicycling', 'fishing and hunting', 'dancing', 'walking',
                            'running',
                            'self care', 'home repair', 'volunteer activities', 'music playing', 'transportation']
        self._load_data()
        self.initial_class_counts = Counter(self.labels)
        self.class_to_index = {cls_name: i for i, cls_name in enumerate(self.class_names)}  # Corrected mapping
        self.index_to_class = {i: cls_name for i, cls_name in enumerate(self.class_names)}

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert('RGB')

        label_str = self.labels[idx]  # Get the string label
        label_int = self.class_to_index[label_str]  # Convert to integer using the mapping
        label = torch.tensor(label_int, dtype=torch.long)

        if self.transform:
            image = self.transform(image)

        return image, label
    def get_num_classes(self):
        """Returns the number of unique classes in the dataset after filtering."""
        return len(set(self.labels))

    def _load_data(self):
        """Loads image file paths and their corresponding string labels."""
        for img_name in os.listdir(self.img_dir):
            if img_name.endswith('.jpg'):
                img_id = os.path.splitext(img_name)[0]
                label_index = self.labels_df[self.labels_df['img_id'].astype(str) == img_id]['target_feature'].values[0]
                label_str = self.class_names[label_index]  # Get the string label
                self.image_files.append(img_name)
                self.labels.append(label_str)  # Store string label

    def print_class_distribution(self):
        """Prints the number of samples and proportion for each class in the dataset."""
        class_counts = Counter(self.labels)
        total_samples = len(self.labels)
        print("Class Distribution:")
        for class_name in self.class_names:
            count = class_counts.get(class_name, 0)  # Get count using the class name
            proportion = count / total_samples if total_samples > 0 else 0
            print(f"  Class '{class_name}': {count} samples ({proportion:.4f})")

    def filter_by_min_threshold(self, min_threshold):
        """Filters the dataset to keep only classes with a proportion of samples
        greater than or equal to the min_threshold.
        """
        total_samples = len(self.labels)
        filtered_image_files = []
        filtered_labels_strings = []
        excluded_classes_log = {}
        original_numerical_labels = list(self.labels)  # Keep the numerical labels for iteration

        class_counts = Counter()
        for label_index in original_numerical_labels:
            class_name = self.class_names[label_index]
            class_counts[class_name] += 1

        class_proportions = {}
        for class_name in self.class_names:
            count = class_counts.get(class_name, 0)
            class_proportion = count / total_samples if total_samples > 0 else 0
            class_proportions[class_name] = class_proportion

        indices_to_keep = []
        for idx, label_index in enumerate(original_numerical_labels):
            class_name = self.class_names[label_index]  # Transform index to name
            if class_proportions[class_name] >= min_threshold:
                indices_to_keep.append(idx)

        self.image_files = [self.image_files[i] for i in indices_to_keep]
        # Now, update self.labels to contain the *numerical* labels of the kept samples
        self.labels = [self.labels[i] for i in indices_to_keep]

        # Identify and log excluded classes
        for i, class_name in enumerate(self.class_names):
            class_proportion = class_proportions.get(class_name, 0)
            original_count = class_counts.get(class_name, 0)
            if class_proportion < min_threshold and original_count > 0:
                excluded_classes_log[class_name] = original_count

        print(f"Filtering by minimum threshold ({min_threshold:.4f}):")
        if excluded_classes_log:
            for cls, count in excluded_classes_log.items():
                proportion = class_proportions[cls]
                print(f"  Excluded images of class '{cls}' with {count} samples (proportion: {proportion:.4f}).")
        else:
            print("  No classes excluded based on the minimum proportion threshold.")
        print(f"Dataset size reduced to {len(self.labels)} from {total_samples} samples initially.")
        print(f"  Number of images after filtering: {len(self.image_files)}")
        print(f"  Number of labels after filtering: {len(self.labels)}")
        print(f"  Unique labels after filtering: {len(set(self.labels))}")

    def filter_by_classes(self, classes_to_exclude):
        """Filters the dataset to remove samples of specified classes and logs excluded classes."""
        original_length = len(self.labels)
        filtered_image_files = []
        filtered_labels = []
        excluded_classes_log = {}

        for img_file, label_str in zip(self.image_files, self.labels):
            if label_str not in classes_to_exclude:
                filtered_image_files.append(img_file)
                filtered_labels.append(label_str)
            elif label_str in classes_to_exclude and label_str not in excluded_classes_log:
                excluded_classes_log[label_str] = self.labels.count(label_str)

        self.image_files = filtered_image_files
        self.labels = filtered_labels

        # Rebuild class_names and class_to_index
        self.class_names = sorted(list(set(self.labels)))
        self.class_to_index = {cls_name: i for i, cls_name in enumerate(self.class_names)}
        self.index_to_class = {i: cls_name for i, cls_name in enumerate(self.class_names)}
        print(f"Number of classes after filtering: {len(self.class_names)}")

        print(f"Filtering by explicitly excluded classes ({classes_to_exclude}):")
        if excluded_classes_log:
            for cls, count in excluded_classes_log.items():
                print(f"  Excluded images of class '{cls}' with {count} samples.")
        else:
            print("  No specified classes found in the dataset to exclude.")
        print(f"Dataset size reduced to {len(self.labels)} from {original_length} samples initially.")

In [49]:
# dataloader

class dataloader():
    def __init__(self):
        data_dir = "PATH TO YOUR DATA"
        transforms = get_transforms()

        full_dataset = PeopleDataset(data_dir, transform=transforms)

        # Example of using filter_by_classes
        classes_to_exclude = ['inactivity quiet/light', 'religious activities', 'running', 'self care', 'volunteer activities', 'transportation']
        print("Excluding selected classes")
        full_dataset.filter_by_classes(classes_to_exclude)
        full_dataset.print_class_distribution()
        print(f"Number of classes: {full_dataset.get_num_classes()}")
        print("Setting up data loaders with batch_size=32...")

        train_set, valid_set = split_dataset(full_dataset, valid_ratio=0.2)

        batch_size = 32
        train_loader, valid_loader = setup_data_loaders(
            batch_size=batch_size,
            train_set=train_set,
            valid_set=valid_set
        )

        print_batch_shape(train_loader, "Train")
        if valid_loader:
            print_batch_shape(valid_loader, "Validation")
    
    
def get_class_weights(dataset):
    targets = [dataset[i][1].item() for i in range(len(dataset))]
    class_counts = Counter(targets)
    total_samples = len(targets)
    num_classes = len(class_counts)

    class_weights = {cls: total_samples / count for cls, count in class_counts.items()}
    sample_weights = [class_weights[label] for label in targets]

    return torch.DoubleTensor(sample_weights)


def get_transforms(augmentation_type=None):
    print(f"Augmentation type: {augmentation_type}")
    if augmentation_type == "basic":
        return basic_augmentation
    elif augmentation_type == "advanced":
        return advanced_augmentation
    else:
        return basic_transformation


def setup_data_loaders(batch_size, train_set, valid_set=None, num_workers=4, use_sampler=False):
    if use_sampler:
        sample_weights = get_class_weights(train_set)
        sampler = WeightedRandomSampler(
            weights=sample_weights,
            num_samples=len(sample_weights),
            replacement=True
        )
        train_loader = DataLoader(
            train_set,
            batch_size=batch_size,
            sampler=sampler,
            num_workers=num_workers
        )
    else:
        train_loader = DataLoader(
            train_set,
            batch_size=batch_size,
            shuffle=True,
            num_workers=num_workers
        )

    valid_loader = DataLoader(
        valid_set,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers
    ) if valid_set is not None else None

    return train_loader, valid_loader


def print_batch_shape(data_loader: DataLoader, loader_type: str, ):
    valid_batch = next(iter(data_loader))
    images_valid, labels_valid = valid_batch
    print(f"{loader_type} batch shape: {images_valid.shape}")


def split_dataset(dataset, valid_ratio=0.25):
    total_size = len(dataset)
    valid_size = int(total_size * valid_ratio)
    train_size = total_size - valid_size
    return random_split(dataset, [train_size, valid_size])

In [50]:
# plotting

def plot_metric(metric_name, subplot_num, epochs, train_metric_value, valid_metric_value, nrows, ncols, is_ylim=False):
    plt.subplot(nrows, ncols, subplot_num)

    # Check if the train_metric_value or valid_metric_value are empty before accessing
    if not train_metric_value:
        print(f"Warning: Missing data for train {metric_name}. Skipping plot for this metric.")
        return
    if not valid_metric_value:
        print(f"Warning: Missing data for valid {metric_name}. Skipping plot for this metric.")
        return

    # Convert tensors to numpy arrays if needed
    def convert_to_numpy(values):
        # If values are tensors, reduce them to a scalar (e.g., mean) or convert to numpy array
        if isinstance(values[0], Tensor):
            # Convert tensor to numpy array (taking mean if needed)
            values = [val.mean().item() if isinstance(val, Tensor) else val for val in values]
        return values

    train_metric_value = convert_to_numpy(train_metric_value)
    valid_metric_value = convert_to_numpy(valid_metric_value)

    # Check if the lengths of epochs and metric values match
    if len(train_metric_value) != len(epochs) or len(valid_metric_value) != len(epochs):
        print(f"Error: The number of epochs ({len(epochs)}) does not match the number of values for {metric_name}. "
              f"Skipping plot for this metric.")
        return

    # Plot the metric
    plt.plot(epochs, train_metric_value, label=f"Train {metric_name}", color='blue')
    plt.plot(epochs, valid_metric_value, label=f"Valid {metric_name}", color='orange')
    plt.title(metric_name)
    plt.xlabel("Epochs")
    plt.ylabel(metric_name)

    # If y-limits are needed, set them
    if is_ylim:
        plt.ylim(0, 1)

    plt.legend()
    plt.gca().xaxis.set_major_locator(ticker.MaxNLocator(integer=True))


def plot_metrics(train_metrics_history: defaultdict, valid_metrics_history: defaultdict,
                 metrics_to_plot: List[str], save_path: str = None):
    if "loss" not in train_metrics_history or not train_metrics_history["loss"]:
        print("Error: No training loss data found!")
        return

    epochs = range(1, len(train_metrics_history["loss"]) + 1)
    metrics_to_plot_without_loss = [metric for metric in metrics_to_plot if metric != "loss"]
    total_metrics = len(metrics_to_plot_without_loss) + 1

    ncols = math.ceil(total_metrics ** 0.5)
    nrows = math.ceil(total_metrics / ncols)

    plt.figure(figsize=(4 * ncols, 4 * nrows))

    train_loss = train_metrics_history.get("loss", [])
    valid_loss = valid_metrics_history.get("loss", [])
    plot_metric("Loss", 1, epochs, train_loss, valid_loss, nrows, ncols)

    for idx, metric_name in enumerate(metrics_to_plot_without_loss, start=2):
        train_metric_value = train_metrics_history.get(metric_name, [])
        valid_metric_value = valid_metrics_history.get(metric_name, [])
        plot_metric(metric_name, idx, epochs, train_metric_value, valid_metric_value, nrows, ncols, is_ylim=True)

    plt.tight_layout()
    if save_path:
        os.makedirs(save_path, exist_ok=True)
        plt.savefig(os.path.join(save_path, f"metrics.png"), dpi=300, bbox_inches='tight')
        print("saved metrics")
    plt.close()


def plot_metric_and_loss(train_metrics_history: defaultdict, valid_metrics_history: defaultdict, metric_to_plot: str):
    # Ensure the 'loss' exists and is not empty for defining epochs
    if "loss" not in train_metrics_history or not train_metrics_history["loss"]:
        print("Error: No training loss data found!")
        return

    epochs = range(1, len(train_metrics_history["loss"]) + 1)

    # Check if the given metric is valid
    if metric_to_plot not in train_metrics_history or metric_to_plot not in valid_metrics_history:
        print(f"Error: The metric '{metric_to_plot}' is not found in the provided data.")
        return

    # Set up the plot size
    plt.figure(figsize=(10, 5))

    # Plot Train loss and Valid loss
    train_loss = train_metrics_history.get("loss", [])
    valid_loss = valid_metrics_history.get("loss", [])

    plot_metric("Loss", 1, epochs, train_loss, valid_loss, nrows=1, ncols=2)

    # Plot the specified additional metric (e.g., accuracy, precision, etc.)
    train_metric_value = train_metrics_history.get(metric_to_plot, [])
    valid_metric_value = valid_metrics_history.get(metric_to_plot, [])
    plot_metric(metric_to_plot, 2, epochs, train_metric_value, valid_metric_value, nrows=1, ncols=2, is_ylim=True)

    plt.tight_layout()
    plt.show()


def visualize_predictions(model, valid_loader, device, class_names, num_images=15):
    model.eval()
    images, labels = next(iter(valid_loader))
    images, labels = images.to(device), labels.to(device)

    with no_grad():
        outputs = model(images)
        _, predicted = torch_max(outputs, 1)

    random_indices = random_sample(range(len(images)), num_images)
    fig, axes = plt.subplots(3, 5, figsize=(8, 5))
    axes = axes.flatten()

    for i, idx in enumerate(random_indices):
        ax = axes[i]
        ax.imshow(images[idx].cpu().numpy().transpose(1, 2, 0))

        title = f"Pred: {class_names[predicted[idx].item()]}\nTrue: {class_names[labels[idx].item()]}"
        ax.set_title(title)
        ax.axis('off')

    plt.tight_layout()
    plt.show()


def show_first_images(dataset):
    plt.figure(figsize=(8, 6))
    for i in range(12):
        image, label = dataset[i]
        # Если использовались torchvision трансформации, то нужно преобразовать обратно в PIL или numpy
        if isinstance(image, Tensor):
            image = image.permute(1, 2, 0).numpy()  # CHW -> HWC
        plt.subplot(3, 4, i + 1)
        plt.imshow(image)
        plt.title(f"Label: {label.item() if isinstance(label, Tensor) else label}")
        plt.axis("off")
    plt.tight_layout()
    plt.show()


def plot_metrics_per_class(model, data_loader, device, class_names, save_path: str = None):
    model.eval()
    all_preds = []
    all_targets = []

    with no_grad():
        for batch_idx, (data, target) in enumerate(data_loader):
            data, target = data.to(device), target.to(device)
            output = model(data)
            preds = argmax(output, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(target.cpu().numpy())

    if len(all_preds) == 0 or len(all_targets) == 0:
        print("Error: Empty predictions or targets.")
        return

    num_classes = len(class_names)
    class_accuracies = []
    class_precisions = []
    class_f1s = []

    for class_idx in range(num_classes):
        true_labels = np.array(all_targets)
        pred_labels = np.array(all_preds)
        class_mask = true_labels == class_idx

        if np.sum(class_mask) == 0:
            acc = prec = f1 = 0
        else:
            relevant_preds = pred_labels[class_mask]
            relevant_trues = true_labels[class_mask]
            acc = accuracy_score(relevant_trues, relevant_preds)
            prec = precision_score(true_labels, pred_labels, labels=[class_idx], average='macro', zero_division=0)
            f1 = f1_score(true_labels, pred_labels, labels=[class_idx], average='macro', zero_division=0)

        class_accuracies.append(acc)
        class_precisions.append(prec)
        class_f1s.append(f1)

    x = np.arange(num_classes)
    width = 0.25

    plt.figure(figsize=(16, 8))
    plt.bar(x - width, class_accuracies, width, label='Accuracy')
    plt.bar(x, class_precisions, width, label='Precision')
    plt.bar(x + width, class_f1s, width, label='F1 Score')

    plt.xlabel('Class')
    plt.ylabel('Score')
    plt.title('Per-Class Accuracy, Precision, and F1 Score')
    plt.xticks(x, class_names, rotation=45, ha='right')
    plt.ylim(0, 1.05)
    plt.legend()
    plt.tight_layout()
    plt.grid(True, linestyle='--', alpha=0.5)

    if save_path:
        os.makedirs(save_path, exist_ok=True)
        plt.savefig(os.path.join(save_path, f"distribution_per_classes.png"),
                    dpi=300,
                    bbox_inches='tight')
        print("saved distribustion")
    plt.close()


In [51]:
# logging

def log_iteration_loss(engine):
    print(f"Epoch[{engine.state.epoch}] - Iter[{engine.state.iteration}]: loss = {engine.state.output}")


def run_evaluators_on_epoch(train_evaluator, valid_evaluator, train_loader, valid_loader):
    train_evaluator.run(train_loader)
    valid_evaluator.run(valid_loader)


def log_and_save_epoch_results(engine, label, metrics_history, silent=False):
    metrics = engine.state.metrics
    metrics_items = metrics.items()
    result = ', '.join([
        f"{m} = {v.mean().item():.4f}" if isinstance(v, Tensor) and v.numel() > 1 else f"{m} = {v.item():.4f}"
        if isinstance(v, Tensor) else f"{m} = {v:.4f}"
        for m, v in metrics_items
    ])
    if not silent:
        print(f"{label}: {result}")

    for metric_name, value in metrics_items:
        metrics_history[metric_name].append(value)


def setup_metrics_history():
    train_metrics_history = defaultdict(list)
    valid_metrics_history = defaultdict(list)
    return train_metrics_history, valid_metrics_history


def add_metrics_to_history(metrics_history, metrics_data: dict):
    metrics_history['loss'].append(metrics_data["loss"])
    metrics_history['accuracy'].append(metrics_data["accuracy"])
    metrics_history['precision'].append(metrics_data["precision"])
    metrics_history['recall'].append(metrics_data["recall"])
    metrics_history['f1'].append(metrics_data["f1"])
    return metrics_history


def print_metrics(data_type: str, metrics_dict: dict):
    """
    :param data_type: "Train", "Valid" or "Test"
    :param metrics_dict: dict of metrics values
    """

    loss = metrics_dict["loss"]
    accuracy = metrics_dict["accuracy"]
    precision = metrics_dict["precision"]
    recall = metrics_dict["recall"]
    f1 = metrics_dict["f1"]

    print(f"{data_type} - Loss: {loss :.4f}, Acc: {accuracy :.2f}%, "
          f"Precision: {precision :.4f}, Recall: {recall :.4f}, F1: {f1 :.4f}")


def print_epoch_summary(epoch: int, train_metrics_dict: dict, valid_metrics_dict: dict):
    print(f"\nEpoch {epoch + 1} Summary:")
    print_metrics("Train", train_metrics_dict)
    if valid_metrics_dict:
        print_metrics("Valid", valid_metrics_dict)


def setup_event_handlers(trainer, optimizer,
                         train_evaluator, valid_evaluator,
                         train_metrics_history, valid_metrics_history,
                         train_loader, valid_loader,
                         silent=False, log_interval=100):
    if not silent:
        trainer.add_event_handler(Events.ITERATION_COMPLETED(every=log_interval), log_iteration_loss)

        def log_lr():
            for param_group in optimizer.param_groups:
                print(f"Optimizer learning rate = {param_group['lr']}")
            print()

        valid_evaluator.add_event_handler(Events.COMPLETED, log_lr)

    trainer.add_event_handler(Events.EPOCH_COMPLETED,
                              run_evaluators_on_epoch,
                              train_evaluator, valid_evaluator,
                              train_loader, valid_loader)

    train_evaluator.add_event_handler(Events.EPOCH_COMPLETED,
                                      log_and_save_epoch_results,
                                      label="Train", metrics_history=train_metrics_history, silent=silent)

    valid_evaluator.add_event_handler(Events.EPOCH_COMPLETED,
                                      log_and_save_epoch_results,
                                      label="Valid", metrics_history=valid_metrics_history, silent=silent)

    scheduler = ReduceLROnPlateauScheduler(optimizer, metric_name="loss", factor=0.5, patience=1, threshold=0.05)
    valid_evaluator.add_event_handler(Events.COMPLETED, scheduler)


def save_best_models(current_metrics, model, model_name, best_loss, best_f1, save_dir):
    save_info = []
    os.makedirs(save_dir, exist_ok=True)
    if current_metrics['loss'] < best_loss:
        best_loss = current_metrics['loss']
        loss_path = os.path.join(save_dir, f"{model_name}_best_loss.pt")
        torch.save(model.state_dict(), loss_path)
        save_info.append(f"Loss: {best_loss:.4f}")

    if current_metrics['f1'] > best_f1:
        best_f1 = current_metrics['f1']
        f1_path = os.path.join(save_dir, f'{model_name}_best_f1.pt')
        torch.save(model.state_dict(), f1_path)
        save_info.append(f"F1: {best_f1:.4f}")

    if save_info:
        print(f"Saved new best model(s) - {' | '.join(save_info)}")

    return best_loss, best_f1


In [52]:
# engine

def setup_trainer(model, optimizer, criterion, device):
    trainer = create_supervised_trainer(model, optimizer, criterion, device=device)
    return trainer


def setup_evaluators(model, criterion, device):
    precision = Precision()
    recall = Recall()
    f1 = Fbeta(beta=1.0, average=False, precision=precision, recall=recall)

    metrics = {'accuracy': Accuracy(),
               'precision': precision,
               'recall': recall,
               'f1': f1,
               "loss": Loss(criterion)}

    train_evaluator = create_supervised_evaluator(model, metrics=metrics, device=device)
    valid_evaluator = create_supervised_evaluator(model, metrics=metrics, device=device)
    return train_evaluator, valid_evaluator


def evaluate_model(model, test_loader, criterion, device, out_for_table=False):
    """Оценивает модель на тестовом наборе данных после обучения."""
    metrics = {"accuracy": Accuracy(), "loss": Loss(criterion)}
    test_evaluator = create_supervised_evaluator(model, metrics=metrics, device=device)
    test_evaluator.run(test_loader)
    metrics = test_evaluator.state.metrics

    if out_for_table:
        params_count = sum(p.numel() for p in model.parameters())
        print(f"| {params_count} | {metrics['accuracy']:.4f} | {metrics['loss']:.4f} |")
    else:
        print(f"Test Results: Accuracy = {metrics['accuracy']:.4f}, Loss = {metrics['loss']:.4f}")


def calculate_metrics(preds, targets):
    """Calculate precision, recall, f1 score"""
    preds = np.array(preds)
    targets = np.array(targets)

    precision = precision_score(targets, preds, average='weighted', zero_division=0)
    recall = recall_score(targets, preds, average='weighted', zero_division=0)
    f1 = f1_score(targets, preds, average='weighted', zero_division=0)

    return precision, recall, f1


def train_step(model, data, target, criterion, optimizer, device):
    model.train()
    data, target = data.to(device), target.to(device)
    output = model(data)
    loss = criterion(output, target)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    preds = torch.argmax(output, dim=1)
    return loss.item(), preds.cpu().numpy(), target.cpu().numpy()


def train_epoch_and_get_metrics_dict(model, train_loader, criterion, optimizer, device, epoch, num_epochs):
    model.train()
    train_loss = 0
    all_train_preds = []
    all_train_targets = []
    train_iterator = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{num_epochs} Training", unit="batch")

    for batch_idx, (data, target) in enumerate(train_iterator):
        loss, preds, targets = train_step(model, data, target, criterion, optimizer, device)
        train_loss += loss
        all_train_preds.extend(preds)
        all_train_targets.extend(targets)
        train_iterator.set_postfix(loss=loss)

    avg_loss = train_loss / len(train_loader)
    accuracy = 100. * np.sum(np.array(all_train_preds) == np.array(all_train_targets)) / len(all_train_targets)
    precision, recall, f1 = calculate_metrics(all_train_preds, all_train_targets)

    metrics_dict = create_metrics_dict(avg_loss, accuracy, precision, recall, f1)

    return metrics_dict


def calculate_epoch_metrics(model, valid_loader, criterion, device):
    """Validates one epoch of model"""
    model.eval()

    val_loss = 0
    all_val_preds = []
    all_val_targets = []
    valid_iterator = tqdm(valid_loader, desc="Validation", unit="batch")

    with torch.no_grad():
        for data, target in valid_iterator:
            data, target = data.to(device), target.to(device)
            output = model(data)
            loss = criterion(output, target).item()
            val_loss += loss
            preds = torch.argmax(output, dim=1).cpu().numpy()
            all_val_preds.extend(preds)
            all_val_targets.extend(target.cpu().numpy())
            valid_iterator.set_postfix(val_loss=loss / len(valid_loader))

    avg_loss = val_loss / len(valid_loader)
    accuracy = 100. * np.sum(np.array(all_val_preds) == np.array(all_val_targets)) / len(all_val_targets)
    precision, recall, f1 = calculate_metrics(all_val_preds, all_val_targets)

    metrics_dict = create_metrics_dict(avg_loss, accuracy, precision, recall, f1)
    return metrics_dict


def create_metrics_dict(avg_loss, accuracy, precision, recall, f1):
    metrics_dict = {
        "loss": avg_loss,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }
    return metrics_dict


In [53]:
# train function (from file src/launcher/traner.py)

def train_model(config, model_class, class_exclusion_threshold=None, classes_to_exclude=None):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}\n")

    """Preparing the data"""
    train_transforms = get_transforms(augmentation_type=config.TRAIN_AUGMENTATION_TYPE)
    valid_transforms = get_transforms(augmentation_type=config.VALID_AUGMENTATION_TYPE)

    print("Loading the dataset...")
    full_dataset = PeopleDataset(config.PATH_TO_DATA)
    full_dataset.print_class_distribution()

    if class_exclusion_threshold:
        print("Removing rare classes")
        # Option 1: Filter by minimum threshold of class in dataset
        full_dataset.filter_by_min_threshold(min_threshold=class_exclusion_threshold)

    if classes_to_exclude:
        print("Excluding selected classes")
        # Option 2: Filter by explicitly excluding class names
        full_dataset.filter_by_classes(classes_to_exclude=classes_to_exclude)

    if classes_to_exclude or class_exclusion_threshold:
        # Rebuild class_to_index AFTER filtering
        full_dataset.class_names = sorted(
            list(set(full_dataset.labels)))  # Get unique remaining labels (which are strings) and sort them
        full_dataset.class_to_index = {cls_name: i for i, cls_name in enumerate(full_dataset.class_names)}
        print(f"Number of classes after filtering: {len(full_dataset.class_names)}")  # Verify the number of classes

    train_set, valid_set = split_dataset(full_dataset, valid_ratio=0.2)
    train_set.dataset.transform = train_transforms
    valid_set.dataset.transform = valid_transforms

    # Showing first 12 images after transforming them
    # plotting.show_first_images(full_dataset)

    print(f"Setting up data loaders with batch_size={config.BATCH_SIZE}...")
    train_loader, valid_loader = setup_data_loaders(
        batch_size=config.BATCH_SIZE,
        train_set=train_set,
        valid_set=valid_set
    )

    """Training setup"""
    num_classes = len(full_dataset.class_names)  # Use the updated class_names
    print(f"Number of classes: {num_classes}")

    model = model_class(num_classes=num_classes)
    model.to(device)
    summary(model, (3, 288, 512))
    print("\n")

    train_indices = train_set.indices
    train_targets = [full_dataset[idx][1] for idx in train_indices]
    class_counts = [train_targets.count(i) for i in range(num_classes)]
    class_counts = [c if c != 0 else 1 for c in class_counts]
    class_weights = 1.0 / torch.tensor(class_counts, dtype=torch.float)
    class_weights = class_weights / class_weights.sum()
    class_weights = class_weights.to(device)

    optimizer = AdamW(model.parameters(), lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY)

    if config.SCHEDULER == 'CosineAnnealingWarmRestarts':
        scheduler = CosineAnnealingWarmRestarts(optimizer=optimizer, T_0=25, eta_min=1e-6)
    elif config.SCHEDULER == 'CosineAnnealingLR':
        scheduler = CosineAnnealingLR(optimizer=optimizer, T_max=config.NUM_EPOCHS, eta_min=0)

    criterion = CrossEntropyLoss(weight=class_weights.to(device))

    print("Initializing trainer and evaluators...")
    trainer = setup_trainer(model, optimizer, criterion, device)
    train_evaluator, valid_evaluator = setup_evaluators(model, criterion, device)

    train_metrics_history, valid_metrics_history = setup_metrics_history()

    best_valid_loss = float('inf')
    best_valid_f1 = 0.0
    model_name = model.__class__.__name__

    print(f"\nStarting training for {config.NUM_EPOCHS} epochs...")

    for epoch in range(config.NUM_EPOCHS):
        print(f"\nEpoch {epoch + 1}/{config.NUM_EPOCHS}")

        train_metrics_dict = train_epoch_and_get_metrics_dict(model, train_loader, criterion, optimizer, device, epoch,
                                                              config.NUM_EPOCHS)
        scheduler.step()
        add_metrics_to_history(train_metrics_history, train_metrics_dict)

        valid_metrics_dict = {}
        if valid_loader:
            valid_metrics_dict = calculate_epoch_metrics(model, valid_loader, criterion, device)
            add_metrics_to_history(valid_metrics_history, valid_metrics_dict)
            best_valid_loss, best_valid_f1 = save_best_models(
                current_metrics=valid_metrics_dict,
                model=model,
                model_name=model_name,
                best_loss=best_valid_loss,
                best_f1=best_valid_f1,
                save_dir=config.SAVE_DIR
            )

        print_epoch_summary(epoch, train_metrics_dict, valid_metrics_dict)

    """Results visualization"""
    print("\nTraining completed!")
    print(f"Results location: {config.RESULT_DIR}")
    print("Saving results...")
    metrics_to_plot = ['accuracy', 'precision', 'recall', 'f1', 'loss']
    plot_metrics(train_metrics_history, valid_metrics_history, metrics_to_plot, save_path=config.RESULT_DIR)

    # To plot loss and one metric
    # plot_metric_and_loss(train_metrics_history, valid_metrics_history, "accuracy")

    class_names = full_dataset.class_names  # Use the updated class_names
    # plotting.visualize_predictions(model, valid_loader, device, class_names)

    print(f"\nPlotting metrics per class...")
    plot_metrics_per_class(model, valid_loader, device, class_names, save_path=config.RESULT_DIR)

    # evaluate_model(model, test_loader, criterion, device)


In [54]:
class PoseCNNsc_13_24_35(nn.Module):
    def __init__(self, num_classes=20):
        super(PoseCNNsc_13_24_35, self).__init__()

        def conv_block(in_channels, out_channels, use_dropout=False):
            layers = [
                nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2)
            ]
            if use_dropout:
                layers.append(nn.Dropout2d(0.2))
            return nn.Sequential(*layers)

        self.conv1 = conv_block(3, 32)  # Output: (32, H/2, W/2)
        self.conv2 = conv_block(32, 64)  # Output: (64, H/4, W/4)
        self.conv3 = conv_block(64, 128, use_dropout=False)  # Output: (128, H/8, W/8)
        self.conv4 = conv_block(128, 256, use_dropout=True)  # Output: (256, H/16, W/16)
        self.conv5 = conv_block(256, 512, use_dropout=True)  # Output: (512, H/32, W/32)

        # Skip connection from conv1 to conv3
        self.skip1_proj = nn.Sequential(
            nn.Conv2d(32, 128, kernel_size=1),
            nn.MaxPool2d(kernel_size=4, stride=4)  # Downsample H/2 -> H/8, W/2 -> W/8
        )

        # Skip connection from conv2 to conv4
        self.skip2_proj = nn.Sequential(
            nn.Conv2d(64, 256, kernel_size=1),
            nn.MaxPool2d(kernel_size=2, stride=2)  # Downsample H/4 -> H/8, W/4 -> W/8
        )

        # Skip connection from conv3 to conv5
        self.skip3_proj = nn.Sequential(
            nn.Conv2d(128, 512, kernel_size=1),
            nn.MaxPool2d(kernel_size=4, stride=4)  # Downsample H/8 -> H/32, W/8 -> W/32
        )

        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x1 = self.conv1(x)  # (32, H/2, W/2)
        x2 = self.conv2(x1)  # (64, H/4, W/4)
        x3 = self.conv3(x2)  # (128, H/8, W/8)

        # Skip connection 1 -> 3
        skip1 = self.skip1_proj(x1)  # (128, H/8, W/8)
        x3 = x3 + skip1

        # Skip connection 2 -> 4
        skip2 = self.skip2_proj(x2)  # (256, H/8, W/8)
        x4 = self.conv4(x3)  # (256, H/16, W/16)
        skip2_downsampled = nn.MaxPool2d(kernel_size=2, stride=2)(skip2)  # (256, H/16, W/16)
        x4 = x4 + skip2_downsampled

        x5 = self.conv5(x4)  # (512, H/32, W/32)

        # Skip connection 3 -> 5
        skip3 = self.skip3_proj(x3)  # (512, H/32, W/32)
        x5 = x5 + skip3

        x_pooled = self.pool(x5)  # (512, 1, 1)
        return self.classifier(x_pooled)


In [55]:
# simulating config file as class

class Config:
    PATH_TO_DATA = "PATH_TO_YOUR_DATA"
    RESULT_DIR = "RESULT_DIR"
    SAVE_DIR = "WEIGHTS_SAVE_DIR"

    BATCH_SIZE = 32
    LEARNING_RATE = 0.001
    MOMENTUM = 0.9
    NUM_EPOCHS = 75

    SCHEDULER = 'CosineAnnealingLR'

    WEIGHT_DECAY = 3e-3

    TRAIN_AUGMENTATION_TYPE = "basic"
    VALID_AUGMENTATION_TYPE = None

config = Config()

In [56]:
model_to_train = PoseCNNsc_13_24_35

# Call the training function from trainer.py
train_model(config, model_to_train)

Using device: cpu

Augmentation type: basic
Augmentation type: None
Loading the dataset...


FileNotFoundError: [Errno 2] No such file or directory: 'PATH_TO_YOUR_DATA\\train_answers.csv'